# 10.6 Iterating over a set: the `for` statement

In [1]:
# install dependencies
%pip install -q amplpy pandas

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

The examples in the previous section still require that some function be called repeatedly. Python provides looping statements that can do this work automatically, with various options to determine how long the looping should continue.

We begin with the `for` statement, which executes a statement or collection of statements once for each member of some collection of values. To execute our multi-period production problem sensitivity analysis of [Section 10.5](10_5_running_scripts.ipynb) four times, for example, we can use a single `for` statement followed by the function call that we want to repeat. We first set up the model, its data, and the `sens_step()` function as before:

In [2]:
%%writefile steelT.mod

set PROD;     # products
param T > 0;  # number of weeks

param rate {PROD} > 0;          # tons per hour produced
param inv0 {PROD} >= 0;         # initial inventory
param avail {1..T} >= 0;        # hours available in week
param market {PROD,1..T} >= 0;  # limit on tons sold in week

param prodcost {PROD} >= 0;     # cost per ton produced
param invcost {PROD} >= 0;      # carrying cost/ton of inventory
param revenue {PROD,1..T} >= 0; # revenue per ton sold

var Make {PROD,1..T} >= 0;      # tons produced
var Inv {PROD,0..T} >= 0;       # tons inventoried
var Sell {p in PROD, t in 1..T} >= 0, <= market[p,t]; # tons sold

# Total revenue less costs in all weeks
maximize Total_Profit:
   sum {p in PROD, t in 1..T} (revenue[p,t]*Sell[p,t] -
      prodcost[p]*Make[p,t] - invcost[p]*Inv[p,t]);

# Total of hours used by all products
# may not exceed hours available, in each week
subject to Time {t in 1..T}:
   sum {p in PROD} (1/rate[p]) * Make[p,t] <= avail[t];

# Initial inventory must equal given value
subject to Init_Inv {p in PROD}:  Inv[p,0] = inv0[p];

# Tons produced and taken from inventory
# must equal tons sold and put into inventory
subject to Balance {p in PROD, t in 1..T}:
   Make[p,t] + Inv[p,t-1] = Sell[p,t] + Inv[p,t];

Writing steelT.mod


In [3]:
import pandas as pd

steelT_prod = pd.DataFrame(
    [
        ["bands", 200, 10, 10, 2.5],
        ["coils", 140,  0, 11, 3],
    ],
    columns=["PROD", "rate", "inv0", "prodcost", "invcost"],
).set_index("PROD")

steelT_week = pd.DataFrame(
    [
        ["bands", 1, 25, 6000],
        ["bands", 2, 26, 6000],
        ["bands", 3, 27, 4000],
        ["bands", 4, 27, 6500],
        ["coils", 1, 30, 4000],
        ["coils", 2, 35, 2500],
        ["coils", 3, 37, 3500],
        ["coils", 4, 39, 4200],
    ],
    columns=["PROD", "T", "revenue", "market"],
).set_index(["PROD", "T"])

steelT_avail = {1: 40, 2: 40, 3: 32, 4: 40}

In [4]:
def load_steelT(ampl):
    """Read the steelT model and assign its data."""
    ampl.read("steelT.mod")
    ampl.param["T"] = 4
    ampl.param["avail"] = steelT_avail
    ampl.set_data(steelT_prod, "PROD")
    ampl.set_data(steelT_week)

In [5]:
%%writefile steelT_sens.py

def sens_step(ampl, filename="steelT.sens"):
    """Solve steelT, append the results to a file, and add 5 hours to week 3."""
    ampl.solve(solver="highs")
    assert ampl.solve_result == "solved", ampl.solve_result

    with open(filename, "a") as sens:
        print("Total_Profit =", ampl.obj["Total_Profit"].value(), file=sens)
        for name in ["Make", "Sell", "Inv"]:
            print(ampl.var[name].to_pandas().unstack().to_string(), file=sens)
        print(file=sens)

    ampl.param["avail"][3] = ampl.param["avail"][3] + 5

Writing steelT_sens.py


In [6]:
from steelT_sens import sens_step

load_steelT(ampl)

for _ in range(4):
    sens_step(ampl)

HiGHS 1.15.1: optimal solution; objective 515033
16 simplex iterations
0 barrier iterations
HiGHS 1.15.1: optimal solution; objective 532033
0 simplex iterations
0 barrier iterations
HiGHS 1.15.1: optimal solution; objective 549033
0 simplex iterations
0 barrier iterations
HiGHS 1.15.1: optimal solution; objective 565193
1 simplex iterations
0 barrier iterations


The expression after `in` can be any Python iterable: a range of numbers, a list, the keys of a dictionary, or the members of an AMPL set retrieved with `ampl.set[...].to_list()`. (The variable `_` is a Python convention for a loop index that is not used in the body.)

As an alternative to calling a function, we can write the commands directly as the body of the `for` statement, indented beneath it. This approach tends to be clearer and easier to work with, particularly as we make the loop more sophisticated. As a first example, consider how we would go about compiling a table of the objective and the dual value on constraint `Time[3]`, for successive values of `avail[3]`. A script for this purpose is shown in [Figure 10-1](#fig-10-1). After the model and data are loaded, the script provides two Python dictionaries for the table of values:

```python
avail3_obj = {}
avail3_dual = {}
```

For each value `a` of `avail[3]` that we want to try, `avail3_obj[a]` and `avail3_dual[a]` will be the associated objective and dual values. The values to be tried are given by a Python `range`, from the current value of `avail[3]` up to 15 more, in steps of 5:

```python
start = ampl.param["avail"][3]
for a in range(start, start + 16, 5):
```

In [7]:
import pandas as pd

ampl.reset()
load_steelT(ampl)

avail3_obj = {}
avail3_dual = {}

start = ampl.param["avail"][3]
for a in range(start, start + 16, 5):
    ampl.param["avail"][3] = a
    ampl.solve(solver="highs", verbose=False)
    assert ampl.solve_result == "solved", ampl.solve_result
    avail3_obj[a] = ampl.obj["Total_Profit"].value()
    avail3_dual[a] = ampl.con["Time"][3].dual()

pd.DataFrame({"avail3_obj": avail3_obj, "avail3_dual": avail3_dual})

,avail3_obj,avail3_dual
32,515033.0,3400.0
37,532033.0,3400.0
42,549033.0,3400.0
47,565193.0,2980.0


<a id='fig-10-1'></a>**[Figure 10-1](#fig-10-1):** Parameter sensitivity script.

We see here that a `for` loop can be over an arbitrary collection of values, and that the index running over it (`a` in this case) can be used in statements within the loop. After the loop is complete, the desired table is produced by collecting `avail3_obj` and `avail3_dual` into a Pandas DataFrame, as shown at the end of the script in [Figure 10-1](#fig-10-1).

In this example we have suppressed the messages from the solver, by passing `verbose=False` to `solve()`. Because the messages are no longer seen, the `assert` after each solve is all the more important.

Loops are also convenient for generating formatted tables. Suppose that after solving the multi-period production problem, we want to display sales both in tons and as a percentage of the market limit. We could use the `display()` method to produce a table like this:

In [8]:
ampl.param["avail"][3] = 32
ampl.solve(solver="highs", verbose=False)
assert ampl.solve_result == "solved", ampl.solve_result

ampl.display("{t in 1..T, p in PROD} (Sell[p,t], 100*Sell[p,t]/market[p,t])")

:       Sell[p,t] 100*Sell[p,t]/market[p,t]    :=
1 bands    6000            100
1 coils     307              7.675
2 bands    6000            100
2 coils    2500            100
3 bands    1400             35
3 coils    3500            100
4 bands    2000             30.7692
4 coils    4200            100
;



By writing a script that uses Python's formatted string literals (f-strings), we can create a much more effective table. We first retrieve the values that we need into Python dictionaries:

In [9]:
sell = ampl.var["Sell"].to_dict()
market = ampl.param["market"].to_dict()
T = ampl.param["T"].value()

The script to write this table can be as short as two `print` statements, one of them in a `for` loop:

In [10]:
print(f"\n{'SALES'}{'bands':>14}{'coils':>17}")
for t in range(1, T + 1):
    print(f"week {t}"
          f"{sell['bands', t]:9.0f}{100 * sell['bands', t] / market['bands', t]:7.1f}%"
          f"{sell['coils', t]:9.0f}{100 * sell['coils', t] / market['coils', t]:7.1f}%")


SALES         bands            coils
week 1     6000  100.0%      307    7.7%
week 2     6000  100.0%     2500  100.0%
week 3     1400   35.0%     3500  100.0%
week 4     2000   30.8%     4200  100.0%


This approach is undesirably restrictive, however, because it assumes that there will always be two products and that they will always be named `coils` and `bands`. In fact a single f-string cannot write a table in which both the number of rows and the number of columns depend on the data, because the number of entries in it is always fixed.

A more general script for generating the table is shown in [Figure 10-2](#fig-10-2). Each pass through the "outer" loop over the weeks generates one row of the table. Within each pass, an "inner" loop over `PROD` generates the row's product entries. There are more `print` statements than in the previous example, but they are shorter and simpler. We use several statements to write the contents of each line; `print()` normally ends with a newline, but `end=""` tells it to continue on the same line.

In [11]:
PROD = ampl.set["PROD"].to_list()

print("\nSALES", end="")
for p in PROD:
    print(f"{p:>14}   ", end="")
print()
for t in range(1, T + 1):
    print(f"week {t}", end="")
    for p in PROD:
        print(f"{sell[p, t]:9.0f}", end="")
        print(f"{100 * sell[p, t] / market[p, t]:7.1f}%", end="")
    print()


SALES         bands            coils   
week 1     6000  100.0%      307    7.7%
week 2     6000  100.0%     2500  100.0%
week 3     1400   35.0%     3500  100.0%
week 4     2000   30.8%     4200  100.0%


<a id='fig-10-2'></a>**[Figure 10-2](#fig-10-2):** Generating a formatted sales table with nested loops.

Loops can be nested to any depth, and may be iterated over any Python collection. There is one pass through the loop for every member of the collection, in the order of the collection. The members of an AMPL set are retrieved by `to_list()` in AMPL's order for that set: for a set of numbers like `1..T`, or a set declared `ordered` or `circular`, this is the ordering of the set; for an unordered set (like `PROD`) AMPL chooses the order, but the choice is the same every time. Since the [Figure 10-2](#fig-10-2) script retrieves `PROD` only once, all of the entries in one column of the table certainly refer to the same product.

When the data are already in Python, Pandas can also build such a table, with as many rows and columns as the data require:

In [12]:
sales = pd.DataFrame({
    "tons": sell,
    "percent": {key: 100 * sell[key] / market[key] for key in sell},
})
sales.unstack(level=0).round(1)

tons       percent       
  bands coils   bands  coils
1  6000   307   100.0    7.7
2  6000  2500   100.0  100.0
3  1400  3500    35.0  100.0
4  2000  4200    30.8  100.0